# 10_03 What does BERT look at?

Everything so far was built here and trained for seconds. This notebook opens a real pretrained transformer,
**bert-mini**: Google's 4-layer, 4-head BERT with 11 million weights, trained on English books and Wikipedia
and already on this machine. You reproduce one of its attention maps by hand, then go looking for what a
pronoun attends to, and find out how much, and how little, an attention weight tells you.

**How this notebook works.** Every notebook in this course has the same rhythm:

1. **Recall.** Answer from memory before you look anything up. `ask()` tells you at once whether you were right.
2. **Predict, then run.** Before a cell with a surprise in it, write your prediction into `guess()`. The next cell runs the code and `reveal()` compares.
3. **Worked example, then your turn.** One example is done in full; the next, near-identical one has lines marked `# YOUR CODE HERE`.
4. **Check.** A `check_...()` cell tests what you saved, exactly as the checkpoint will, and says what to fix.

Run cells in order with **Shift+Enter**. If you get lost, **Kernel, Restart Kernel and Run All Cells** starts clean.

Running this in Google Colab? This cell sets it up; in CourseLabs it does nothing.

In [ ]:
# Colab setup. In a CourseLabs session this cell does nothing.
import os, sys
if "google.colab" in sys.modules:
    import importlib, importlib.util, subprocess
    LAB, REPO = "lab-nlp-10-attention-is-the-whole-trick", "/content/nlp-course"
    if not os.path.isdir(REPO):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/fenago/nlp-course.git", REPO], check=True)
    os.chdir(f"{REPO}/{LAB}")
    if not os.path.exists("data"):
        os.symlink("../data", "data")
    os.makedirs("out", exist_ok=True)
    os.environ["NLPLAB_DATA"] = f"{REPO}/data"
    sys.path.insert(0, os.getcwd())
    PIP = {'transformers': 'transformers',
           'torch': 'torch',
           'sklearn': 'scikit-learn',
           'numpy': 'numpy',
           'matplotlib': 'matplotlib'}
    missing = [spec for mod, spec in PIP.items() if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
        importlib.invalidate_caches()
    print(f"Ready: {LAB} and its data are in {os.getcwd()}; installed {len(missing)} package(s).")
elif not os.path.isdir("/opt/nlplab/data") and os.path.isdir("data"):
    # A downloaded copy on your own computer: the helpers read data/ from here.
    os.environ["NLPLAB_DATA"] = os.path.abspath("data")

In [ ]:
import json
import math
import os
import torch
import matplotlib.pyplot as plt
from transformers import AutoModel, AutoTokenizer, logging
from nlpcheck import ask, guess, reveal, check_10_03

logging.set_verbosity_error()
logging.disable_progress_bar()
torch.set_num_threads(4)
NAME = "google/bert_uncased_L-4_H-256_A-4"          # bert-mini, from the image's offline copy
tok = AutoTokenizer.from_pretrained(NAME)
bert = AutoModel.from_pretrained(NAME, attn_implementation="eager").eval()
print(f"bert-mini: {sum(p.numel() for p in bert.parameters()):,} weights, "
      f"{bert.config.num_hidden_layers} layers, {bert.config.num_attention_heads} heads per layer")

`attn_implementation="eager"` asks for the plain attention code, the arithmetic of `10_01`, so the model can
hand back its attention weights. The fast fused version gives the same outputs but never forms the weights.

## 1. Recall

**r5.** In `10_02`, why did the transformer without positions fail to reverse the digits?
(a) its output at every position saw the same set of inputs, so first and last were indistinguishable,
(b) it was not trained for long enough, (c) its attention weights did not add up to 1

In [ ]:
ask("r5", "")

**r6.** BERT reads a whole sentence at once, both directions. Does it need a causal mask?
(a) no, it is an encoder, and a causal mask is for a decoder that writes left to right, (b) yes, like every transformer

In [ ]:
ask("r6", "")

## 2. The sentence, and layer 0 by hand

BERT's tokenizer adds `[CLS]` at the start and `[SEP]` at the end. `output_attentions=True` returns one
`(heads, T, T)` map per layer. The worked example checks the first layer's by hand: take what enters layer 0,
apply its query and key matrices, split into 4 heads of 64 numbers, and do exactly what you did in `10_01`.

In [ ]:
sentence = "The phone would not charge because it was broken."
enc = tok(sentence, return_tensors="pt")
toks = tok.convert_ids_to_tokens(enc["input_ids"][0])
with torch.no_grad():
    out = bert(**enc, output_attentions=True, output_hidden_states=True)
att = torch.stack(out.attentions)[:, 0]        # (layers, heads, T, T)
print(toks); print("attention maps:", tuple(att.shape))

h = out.hidden_states[0][0]                    # what enters layer 0: (T, 256)
sa = bert.encoder.layer[0].attention.self
with torch.no_grad():
    q = sa.query(h).view(-1, 4, 64).transpose(0, 1)    # (heads, T, 64)
    k = sa.key(h).view(-1, 4, 64).transpose(0, 1)
    by_hand = torch.softmax(q @ k.transpose(1, 2) / math.sqrt(64), dim=-1)
by_hand_max_diff = float((by_hand - att[0]).abs().max())
print("largest difference between your layer 0 and BERT's:", by_hand_max_diff)

Zero, or within a millionth. A pretrained model with 11 million weights computes its attention the way you did
for three words.

## 3. Where does "it" look in the last layer?

Before you run the next cell: in BERT's last layer, averaged over its four heads, which token does "it" give
the most weight to? (a) "phone", the thing that was broken, (b) "broken", (c) `[SEP]`, the marker at the end

In [ ]:
guess("last_layer", None)   # "a", "b" or "c" 

In [ ]:
it = toks.index("it")
last = att[-1, :, it].mean(0)                  # the last layer, averaged over heads, the row for "it"
top = last.topk(3)
print("top three for 'it' in the last layer:", [(toks[j], round(float(v), 3)) for v, j in zip(top.values, top.indices)])
sink = float(last[toks.index("[SEP]")])
reveal("last_layer", "c" if toks[int(last.argmax())] == "[SEP]" else toks[int(last.argmax())])

`[SEP]`, with about 0.94 of the weight. The last layer of this model sends almost every word's attention to the
end marker. Researchers who studied BERT's attention found the same thing and read it as a way of **doing
nothing**: a softmax has to put its weight somewhere, and a head with nothing useful to add parks it on a token
that carries no information. So a large attention weight is not the same as an important word.

## 4. Your turn: find the head that finds the phone

Somewhere among the 16 heads there is one where "it" does look at "phone". Write the loop: for every layer and
head, read `att[layer, head, it, phone]`, and keep the largest.

In [ ]:
phone = toks.index("phone")
best_layer, best_head, best_weight = None, None, 0.0
# YOUR CODE HERE: loop over the 4 layers and 4 heads, and keep the one with the largest att[layer, head, it, phone]

if best_layer is not None:
    row = att[best_layer, best_head, it]
    print(f"layer {best_layer}, head {best_head}: 'it' gives 'phone' {best_weight:.2f}")
    print("its top three:", [(toks[j], round(float(v), 2)) for v, j in zip(row.topk(3).values, row.topk(3).indices)])
    plt.figure(figsize=(5, 5)); plt.imshow(att[best_layer, best_head], cmap="Greys")
    plt.xticks(range(len(toks)), toks, rotation=90); plt.yticks(range(len(toks)), toks)
    plt.title(f"layer {best_layer}, head {best_head}"); plt.show()

Layer 0, head 3, with 0.38. In that head, "it" reaches for the nearest noun. Is that the model resolving the
pronoun? The next section tests it.

## 5. The test: change one word

Two sentences that differ in their last word. In the first, "it" is the animal; in the second, "it" is the
street. A model that resolves the pronoun through attention should shift the weight from "animal" to "street".
Before you run it: across all 16 heads, how much will the gap between "animal" and "street" change?
(a) a lot, at least 0.2 in some head, (b) hardly at all, under 0.05 in every head

In [ ]:
guess("pair_change", None)   # "a" or "b" 

In [ ]:
def gap(sentence):
    e = tok(sentence, return_tensors="pt")
    t = tok.convert_ids_to_tokens(e["input_ids"][0])
    with torch.no_grad():
        a = torch.stack(bert(**e, output_attentions=True).attentions)[:, 0]
    i = t.index("it")
    return a[:, :, i, t.index("animal")] - a[:, :, i, t.index("street")]     # (layers, heads)

tired = gap("The animal didn't cross the street because it was too tired.")
wide = gap("The animal didn't cross the street because it was too wide.")
pair_change = float((tired - wide).abs().max())
print(f"largest change in any head: {pair_change:.3f}")
reveal("pair_change", "a" if pair_change >= 0.2 else "b")

Hardly at all. Every head attends from "it" in nearly the same way whichever word ends the sentence, so this
model is not resolving the pronoun in its attention, if it resolves it at all. That is the honest lesson of
attention maps: they show where information was **mixed from**, not what the model concluded. A small model
like this one does not solve this puzzle; much larger ones, trained on far more text, do much better.

Reference: [What Does BERT Look At? An Analysis of BERT's Attention (Clark et al., 2019)](https://arxiv.org/abs/1906.04341)

In [ ]:
os.makedirs("out", exist_ok=True)
json.dump({"by_hand_max_diff": by_hand_max_diff, "sink_last_layer": sink, "best_layer": best_layer,
           "best_head": best_head, "best_weight": float(best_weight), "pair_change": pair_change},
          open("out/10_03_bert.json", "w"), indent=1)
check_10_03()

## 6. Exit ticket

**x3.** What does multi-head attention let a model do that one head cannot?
(a) attend to different kinds of relationship at the same time, one per head, (b) read longer sentences,
(c) skip the softmax

In [ ]:
ask("x3", "")

Explain it back: the head you found gives "phone" 0.38 of the weight from "it". Why is that not enough to say
the model knows what "it" refers to? One or two sentences.

*Your explanation:* 